In [2]:
model_name="hf_models/qwen2.5-14b-instruct"


In [3]:
# =========================
# 1. Imports
# =========================
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048

# =========================
# 2. Load Base Model
# =========================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

# =========================
# 3. Add LoRA
# =========================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)

# =========================
# 4. Load Dataset
# =========================
with open("data.json", "r") as f:
    data = json.load(f)

def format_example(example):
    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Input:
Topic: {example["input"]["topic"]}
Question: {example["input"]["question"]}
Choices:
{example["input"]["choices"][0]}
{example["input"]["choices"][1]}
{example["input"]["choices"][2]}
{example["input"]["choices"][3]}

### Response:
{json.dumps(example["output"], ensure_ascii=False)}
"""
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

# =========================
# 5. Training Arguments
# =========================
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_steps=800,            # FAST TRAIN
    warmup_steps=50,
    learning_rate=5e-5,
    logging_steps=20,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    bf16=True,
    output_dir="qwen-aagent-lora",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

# =========================
# 6. Train
# =========================
trainer.train()

# =========================
# 7. Save LoRA
# =========================
model.save_pretrained("qwen-aagent-lora")
tokenizer.save_pretrained("qwen-aagent-lora")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
#### Unsloth: `hf_xet==1.1.10` and `ipykernel>6.30.1` breaks progress bars. Disabling for now in XET.
#### Unsloth: To re-enable progress bars, please downgrade to `ipykernel==6.30.1` or wait for a fix to
https://github.com/huggingface/xet-core/issues/526
INFO 02-15 08:41:06 [__init__.py:225] Automatically detected platform rocm.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.


RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [4]:
ls hf_models


llama-3.1-8b-final/  llama-3.1-8b-instruct/


In [5]:
ls hf_models/qwen2.5-14b-instruct


ls: cannot access 'hf_models/qwen2.5-14b-instruct': No such file or directory


In [6]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-14B-Instruct",
    local_dir="hf_models/qwen2.5-14b-instruct",
    local_dir_use_symlinks=False
)


Ignored error while writing commit hash to /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/refs/main: [Errno 30] Read-only file system: '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct'.
[2026-02-15 08:42:36] WARNING _snapshot_download.py:300: Ignored error while writing commit hash to /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/refs/main: [Errno 30] Read-only file system: '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct'.


'/workspace/AAIPL/hf_models/qwen2.5-14b-instruct'

In [7]:
ls hf_models/qwen2.5-14b-instruct



SyntaxError: invalid decimal literal (4151888292.py, line 1)

In [8]:
# =========================================
# 0. Install if needed (skip if already installed)
# =========================================
!pip install unsloth trl transformers datasets peft huggingface_hub -q

# =========================================
# 1. Download Qwen 2.5-14B Properly
# =========================================
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-14B-Instruct",
    local_dir="hf_models/qwen2.5-14b-instruct",
    local_dir_use_symlinks=False,
    ignore_patterns=["*.msgpack", "*.h5"]  # faster
)

print("Model downloaded.")

# =========================================
# 2. Imports
# =========================================
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import PeftModel

max_seq_length = 2048

# =========================================
# 3. Load Base Model
# =========================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

print("Model loaded.")

# =========================================
# 4. Add LoRA
# =========================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)

print("LoRA added.")

# =========================================
# 5. Load Dataset
# =========================================
with open("data.json", "r") as f:
    data = json.load(f)

def format_example(example):
    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Input:
Topic: {example["input"]["topic"]}
Question: {example["input"]["question"]}
Choices:
{example["input"]["choices"][0]}
{example["input"]["choices"][1]}
{example["input"]["choices"][2]}
{example["input"]["choices"][3]}

### Response:
{json.dumps(example["output"], ensure_ascii=False)}
"""
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

print("Dataset ready.")

# =========================================
# 6. Training Config (Fast Mode)
# =========================================
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_steps=800,
    warmup_steps=50,
    learning_rate=5e-5,
    logging_steps=20,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    bf16=True,
    output_dir="qwen-aagent-lora",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

print("Starting training...")
trainer.train()

# =========================================
# 7. Save LoRA
# =========================================
model.save_pretrained("qwen-aagent-lora")
tokenizer.save_pretrained("qwen-aagent-lora")

print("LoRA saved.")

# =========================================
# 8. Merge LoRA into Base
# =========================================
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

merged_model = PeftModel.from_pretrained(base_model, "qwen-aagent-lora")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained("hf_models/qwen2.5-14b-aagent-final")
tokenizer.save_pretrained("hf_models/qwen2.5-14b-aagent-final")

print("Final merged model saved.")



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


Ignored error while writing commit hash to /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/refs/main: [Errno 30] Read-only file system: '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct'.
[2026-02-15 08:44:04] WARNING _snapshot_download.py:300: Ignored error while writing commit hash to /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct/refs/main: [Errno 30] Read-only file system: '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct'.


Model downloaded.
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Model loaded.


Unsloth 2025.10.9 patched 48 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA added.


FileNotFoundError: [Errno 2] No such file or directory: 'data.json'

In [9]:
import os
print(os.getcwd())


/workspace/AAIPL


In [10]:
os.listdir()


['README.ipynb',
 'agen.yaml',
 'qgen.yaml',
 'utils',
 'assets',
 '.dockerignore',
 '.ipynb_checkpoints',
 'tutorial_config.yaml',
 'tutorial.ipynb',
 'agents',
 'Untitled1.ipynb',
 'hf_models',
 'Untitled.ipynb',
 'outputs',
 'question.json',
 'unsloth_compiled_cache',
 'llama-qagent-lora',
 'presentation.pptx',
 'git.sh']

In [11]:
import json

# Load original dataset
with open("question.json", "r") as f:
    questions = json.load(f)

new_data = []

for q in questions:
    # Extract answer letter (first character before dot)
    answer_letter = q["answer"].split(".")[0].strip()

    new_item = {
        "instruction": "Solve the following reasoning question and return the correct option letter with short logical reasoning in JSON format.",
        "input": {
            "topic": q["topic"],
            "question": q["question"],
            "choices": q["choices"]
        },
        "output": {
            "answer": answer_letter,
            "reasoning": "Provide a clear logical explanation under 90 words based strictly on the given statements."
        }
    }

    new_data.append(new_item)

# Save as data.json
with open("data.json", "w") as f:
    json.dump(new_data, f, indent=2)

print("data.json created successfully.")


data.json created successfully.


In [12]:
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)

print("Model ready.")


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Model ready.


In [13]:
with open("data.json", "r") as f:
    data = json.load(f)

def format_example(example):
    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Input:
Topic: {example["input"]["topic"]}
Question: {example["input"]["question"]}
Choices:
{example["input"]["choices"][0]}
{example["input"]["choices"][1]}
{example["input"]["choices"][2]}
{example["input"]["choices"][3]}

### Response:
{json.dumps(example["output"], ensure_ascii=False)}
"""
    }

from datasets import Dataset
dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

print("Dataset ready.")


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset ready.


In [14]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_steps=500,
    warmup_steps=30,
    learning_rate=5e-5,
    logging_steps=25,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    bf16=True,
    output_dir="qwen-aagent-lora",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=164):   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 5 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,2.252400
50,0.970600
75,0.686500
100,0.607900
125,0.557500
150,0.533900
175,0.518700
200,0.476800
225,0.452700
250,0.429600


TrainOutput(global_step=500, training_loss=0.5493618679046631, metrics={'train_runtime': 309.9674, 'train_samples_per_second': 12.905, 'train_steps_per_second': 1.613, 'total_flos': 6.864205605902746e+16, 'train_loss': 0.5493618679046631, 'epoch': 5.0})

In [15]:
model.save_pretrained("qwen-aagent-lora")
tokenizer.save_pretrained("qwen-aagent-lora")



('qwen-aagent-lora/tokenizer_config.json',
 'qwen-aagent-lora/special_tokens_map.json',
 'qwen-aagent-lora/chat_template.jinja',
 'qwen-aagent-lora/vocab.json',
 'qwen-aagent-lora/merges.txt',
 'qwen-aagent-lora/added_tokens.json',
 'qwen-aagent-lora/tokenizer.json')

In [16]:
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

merged_model = PeftModel.from_pretrained(base_model, "qwen-aagent-lora")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained("hf_models/qwen2.5-14b-aagent-final")
tokenizer.save_pretrained("hf_models/qwen2.5-14b-aagent-final")


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

('hf_models/qwen2.5-14b-aagent-final/tokenizer_config.json',
 'hf_models/qwen2.5-14b-aagent-final/special_tokens_map.json',
 'hf_models/qwen2.5-14b-aagent-final/chat_template.jinja',
 'hf_models/qwen2.5-14b-aagent-final/vocab.json',
 'hf_models/qwen2.5-14b-aagent-final/merges.txt',
 'hf_models/qwen2.5-14b-aagent-final/added_tokens.json',
 'hf_models/qwen2.5-14b-aagent-final/tokenizer.json')

In [17]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-aagent-final",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

model.eval()

prompt = """### Instruction:
Solve the following reasoning question and return the correct option letter with short logical reasoning in JSON format.

### Input:
Topic: Syllogism
Question: Statements: Only wolves are hunters. All hunters are predators. No predator is prey. Conclusions: I. Some wolves are predators. II. No wolf is prey. III. Some wolves being prey is a possibility. Which of the following follows?
Choices:
A. Only I follows
B. Only I and III follow
C. Only II and III follow
D. All follow

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=250,
    temperature=0.0,
    do_sample=False
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

### Instruction:
Solve the following reasoning question and return the correct option letter with short logical reasoning in JSON format.

### Input:
Topic: Syllogism
Question: Statements: Only wolves are hunters. All hunters are predators. No predator is prey. Conclusions: I. Some wolves are predators. II. No wolf is prey. III. Some wolves being prey is a possibility. Which of the following follows?
Choices:
A. Only I follows
B. Only I and III follow
C. Only II and III follow
D. All follow

### Response:
{"answer": "B", "reasoning": "Provide a clear logical explanation under 90 words based strictly on the given statements."}
{"answer": "B", "reasoning": "The statements establish that all wolves fall under hunters, which are entirely contained within predators. Therefore, some wolves being predators is valid (I). However, since no predator is prey, wolves cannot be prey, making II invalid. For III, while it's true that no wolf is prey based on the given structure, the possibility remai

In [18]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.0,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract first JSON only
start = text.find("{")
end = text.find("}") + 1
clean_output = text[start:end]

print(clean_output)


{"answer": "B", "reasoning": "Provide a clear logical explanation under 90 words based strictly on the given statements."}


In [19]:
import json

with open("data.json", "r") as f:
    data = json.load(f)

for item in data:
    topic = item["input"]["topic"]

    if topic == "Syllogism":
        item["output"]["reasoning"] = "From the statements, establish direct logical relationships. Apply inclusion rules carefully and avoid assuming transitivity. Only the conclusion strictly supported by the given statements follows."
    
    elif topic == "Blood Relation":
        item["output"]["reasoning"] = "Construct the family tree using the given relationships. Identify parent-child and sibling links carefully and verify each option against the confirmed structure."
    
    elif topic == "Seating Arrangement":
        item["output"]["reasoning"] = "Arrange individuals according to the positional clues. Fix confirmed placements first and eliminate contradictory possibilities before selecting the correct option."
    
    elif topic == "Mixed Series":
        item["output"]["reasoning"] = "Identify the underlying numerical or positional pattern in the sequence. Test each option against the derived rule and select the one that maintains consistency."

with open("data.json", "w") as f:
    json.dump(data, f, indent=2)

print("Reasoning updated.")


Reasoning updated.


In [20]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    max_seq_length=2048,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [21]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)


In [22]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_steps=1000,          # Your choice
    warmup_steps=50,
    learning_rate=5e-5,
    logging_steps=25,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    bf16=True,
    output_dir="qwen-aagent-lora-1000",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=164):   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 10 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)


Step,Training Loss
25,2.361600
50,1.285700
75,0.710500
100,0.621000
125,0.565900
150,0.539400
175,0.520600
200,0.475800
225,0.446400
250,0.421400


TrainOutput(global_step=1000, training_loss=0.3469005973339081, metrics={'train_runtime': 581.1891, 'train_samples_per_second': 13.765, 'train_steps_per_second': 1.721, 'total_flos': 1.3712045047300915e+17, 'train_loss': 0.3469005973339081, 'epoch': 10.0})

In [23]:
model.save_pretrained("qwen-aagent-lora-1000")
tokenizer.save_pretrained("qwen-aagent-lora-1000")


('qwen-aagent-lora-1000/tokenizer_config.json',
 'qwen-aagent-lora-1000/special_tokens_map.json',
 'qwen-aagent-lora-1000/chat_template.jinja',
 'qwen-aagent-lora-1000/vocab.json',
 'qwen-aagent-lora-1000/merges.txt',
 'qwen-aagent-lora-1000/added_tokens.json',
 'qwen-aagent-lora-1000/tokenizer.json')

In [24]:
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-instruct",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

merged_model = PeftModel.from_pretrained(base_model, "qwen-aagent-lora-1000")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained("hf_models/qwen2.5-14b-aagent-final")
tokenizer.save_pretrained("hf_models/qwen2.5-14b-aagent-final")


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

('hf_models/qwen2.5-14b-aagent-final/tokenizer_config.json',
 'hf_models/qwen2.5-14b-aagent-final/special_tokens_map.json',
 'hf_models/qwen2.5-14b-aagent-final/chat_template.jinja',
 'hf_models/qwen2.5-14b-aagent-final/vocab.json',
 'hf_models/qwen2.5-14b-aagent-final/merges.txt',
 'hf_models/qwen2.5-14b-aagent-final/added_tokens.json',
 'hf_models/qwen2.5-14b-aagent-final/tokenizer.json')

In [25]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/qwen2.5-14b-aagent-final",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

model.eval()

prompt = """### Instruction:
Solve the following reasoning question and return the correct option letter with short logical reasoning in JSON format.

### Input:
Topic: Syllogism
Question: Statements: Only wolves are hunters. All hunters are predators. No predator is prey. Conclusions: I. Some wolves are predators. II. No wolf is prey. III. Some wolves being prey is a possibility. Which of the following follows?
Choices:
A. Only I follows
B. Only I and III follow
C. Only II and III follow
D. All follow

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.0,
    do_sample=False
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

### Instruction:
Solve the following reasoning question and return the correct option letter with short logical reasoning in JSON format.

### Input:
Topic: Syllogism
Question: Statements: Only wolves are hunters. All hunters are predators. No predator is prey. Conclusions: I. Some wolves are predators. II. No wolf is prey. III. Some wolves being prey is a possibility. Which of the following follows?
Choices:
A. Only I follows
B. Only I and III follow
C. Only II and III follow
D. All follow

### Response:
{"answer": "B", "reasoning": "Provide a clear logical explanation under 90 words based strictly on the given statements."}
{"answer": "B", "reasoning": "The structure shows that all wolves fall under hunters, and every hunter is a predator, establishing that some wolves are indeed predators (I follows). Regarding prey, since no predator is prey, wolves cannot be prey due to their classification as predators (II does not follow). However, while wolves are exclusively predators, there i

In [26]:
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.0,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract ONLY first JSON block
import re
match = re.search(r'\{.*?\}', text, re.DOTALL)

if match:
    clean_output = match.group(0)
else:
    clean_output = text

print(clean_output)


{"answer": "B", "reasoning": "Provide a clear logical explanation under 90 words based strictly on the given statements."}
